In [1]:
import pandas as pd

In [2]:
trades = pd.read_excel(
    r"C:\Keshav S\Investements Operations Platform\Trades.xlsx"
)

closing_positions = pd.read_excel(
    r"C:\Keshav S\Investements Operations Platform\Closing_Positions.xlsx"
)

prices = pd.read_excel(
    r"C:\Keshav S\Investements Operations Platform\Prices.xlsx"
)

In [4]:
#Average Price
avg_price = trades.assign(
    notional = trades["Quantity"] * trades["Trade_Price"]
).groupby("Security").agg(
    total_notional=("notional", "sum"),
    total_qty=("Quantity", "sum")
).reset_index()

avg_price["Avg_Price"] = avg_price["total_notional"] / avg_price["total_qty"]

avg_price = avg_price[["Security", "Avg_Price"]]

In [6]:
#Merge data
pnl = closing_positions.merge(avg_price, on="Security", how="left")
pnl = pnl.merge(prices, on="Security", how="left")

In [11]:
#Unrealized P&L
pnl["Unrealized_PnL"] = (
    (pnl["Market_Price"] - pnl["Avg_Price"]) * pnl["Closing_Position"]
)

In [16]:
#P&L %
import numpy as np
pnl["Position_Cost"] = pnl["Avg_Price"] * pnl["Closing_Position"]

In [17]:
import numpy as np

pnl["PnL_%"] = np.where(
    pnl["Position_Cost"] != 0,
    (pnl["Unrealized_PnL"] / pnl["Position_Cost"]) * 100,
    0
)

In [18]:
pnl[["Security", "Position_Cost", "Unrealized_PnL", "PnL_%"]].head()

,Security,Position_Cost,Unrealized_PnL,PnL_%
0,AAPL,2.912433e+06,3.400345e+06,116.752745
1,MSFT,1.517902e+06,2.600237e+05,17.130458
2,GOOG,6.807234e+05,2.197464e+05,32.281306
3,AMZN,1.813231e+06,-1.570132e+06,-86.593058
4,NVDA,2.182248e+06,-1.827308e+06,-83.735112


In [22]:
pnl["Market_Value"] = pnl["Market_Price"] * pnl["Closing_Position"]
summary = pd.DataFrame({
    "Metric": [
        "Total Portfolio Value",
        "Total Unrealized P&L",
        "Total Position Cost"
    ],
    "Value": [
        pnl["Market_Value"].sum(),
        pnl["Unrealized_PnL"].sum(),
        pnl["Position_Cost"].sum()
    ]
})

In [23]:
summary.to_excel(
    r"C:\Keshav S\Investements Operations Platform\PnL_Summary.xlsx",
    index=False
)